# 2.4 Guided Investigation: Tokenization with spaCy

## Text, Tokens & Linguistic Processing

**Estimated time:** 25–35 minutes

In this notebook, you will predict how text will be tokenized, test your predictions, and connect tokens to linguistic information such as lemmas and parts of speech. The goal is guided practice—not a long programming assignment.


## Learning goals

By the end of this investigation, you should be able to:

- explain why tokens are not always the same as whitespace-separated words;
- inspect `text`, `lemma_`, `pos_`, `is_stop`, and `is_punct` for spaCy tokens;
- distinguish stemming from lemmatization; and
- contrast spaCy's word-oriented tokens with modern subword tokens.


## How to use this notebook

For each activity:

1. **Predict** the tokens before running the code.
2. **Run** the code and inspect the output.
3. **Reflect** briefly on what matched—or challenged—your prediction.

You can jot answers in the notebook, discuss them with a partner, or answer aloud as a class.


## Setup

spaCy's tokenizer finds token boundaries. The small English pipeline also adds linguistic annotations, including lemmas, part-of-speech tags, and dependencies.

The next cell installs any missing packages and downloads `en_core_web_sm` if needed. Its first run therefore requires an internet connection; later runs can use the cached packages and model.


In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    "spacy": "spacy>=3.7,<4",
    "snowballstemmer": "snowballstemmer>=2.2,<3",
    "tiktoken": "tiktoken>=0.7,<1",
}

missing = [package for module, package in required_packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import spacy
import tiktoken

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

print(f"spaCy {spacy.__version__}; pipeline: {nlp.pipe_names}")


## 1. Predict Before Running

Consider this sentence:

> We're visiting the U.S. next week!

Before running the next cell, write the token list you expect.

- How many tokens will spaCy produce?
- Will `We're` stay together?
- Will the periods in `U.S.` become separate tokens?
- What will happen to the exclamation point?


In [ ]:
prediction_text = "We're visiting the U.S. next week!"
prediction_doc = nlp(prediction_text)

print([token.text for token in prediction_doc])
print("Token count:", len(prediction_doc))


### Reflect

- Which boundary surprised you most?
- Why might a tokenizer preserve `U.S.` but separate `!`?
- How would a simple call to `prediction_text.split()` differ?


### How spaCy chooses token boundaries

spaCy uses language-specific rules rather than splitting only on spaces. It checks for:

- **prefixes** such as opening quotation marks or `$`;
- **suffixes** such as sentence-final punctuation;
- **infixes** such as some hyphens inside text; and
- **exceptions** for forms such as contractions and abbreviations.

This rule-based, word-oriented approach aims to produce useful units for linguistic analysis. A token may be a word, punctuation mark, number, URL, email address, emoji, or part of a contraction.


## 2. Tokenization challenges

Predict the token list for each example before running the code. Pay special attention to contractions, abbreviations, punctuation inside URLs and email addresses, the decimal point in a price, emoji, and hashtags.


In [ ]:
challenge_texts = {
    "contractions": "I can't believe we're early!",
    "abbreviation": "Let's meet in St. Louis.",
    "URL and email": "Visit https://example.com or email hello@example.com.",
    "price, emoji, hashtag": "The ticket costs $100.60 😊 #NLP.",
}

for label, text in challenge_texts.items():
    tokens = [token.text for token in nlp(text)]
    print(f"{label}:\n  {tokens}\n")


### Reflect

- Which punctuation marks became separate tokens, and which stayed inside a token?
- How did spaCy treat the URL, email address, and `$100.60`?
- Did the emoji and hashtag behave as you expected?
- Which example most clearly shows why whitespace splitting is insufficient?


## 3. Tokens as linguistic objects

A spaCy `Token` stores both its surface form and annotations added by the language pipeline:

- `text`: the exact characters in the document;
- `lemma_`: a context-sensitive dictionary form;
- `pos_`: a coarse-grained part-of-speech tag;
- `is_stop`: whether spaCy marks it as a common stop word; and
- `is_punct`: whether it is punctuation.

These annotations are properties of each token; they do not create additional tokens.


In [ ]:
attribute_doc = nlp("We're visiting the U.S. next week!")

print(f"{'text':<12} {'lemma':<12} {'POS':<8} {'is_stop':<9} {'is_punct':<9}")
print("-" * 55)
for token in attribute_doc:
    print(
        f"{token.text:<12} {token.lemma_:<12} {token.pos_:<8} "
        f"{str(token.is_stop):<9} {str(token.is_punct):<9}"
    )


### Reflect

- Which tokens have a lemma that differs from their text?
- Which contraction piece is marked as a stop word?
- Does `is_punct` depend on the token's meaning or its form?


### A closer look at lemmatization

**Lemmatization** maps an inflected word to a meaningful base form, or *lemma*. It uses linguistic context: for example, forms such as `studies` and `studied` can map to `study`, while `were` maps to `be`.


In [ ]:
lemma_doc = nlp("She studies languages and studied linguistics while they were traveling.")

for token in lemma_doc:
    if token.is_alpha:
        print(f"{token.text:<12} → {token.lemma_}")


### Reflect

- Which word forms were normalized to the same lemma?
- Why could lemmas be useful when counting concepts across a collection of documents?


## 4. Stop words—use with care

Stop words are frequent function words such as articles, auxiliaries, and prepositions. spaCy provides `is_stop` as a convenient default, but removing every marked word is not automatically helpful. For some tasks, words such as `not` carry essential meaning.

Rather than printing or modifying spaCy's full stop-word list, inspect the decision in the context of a sentence.


In [ ]:
stop_doc = nlp("I do not want to miss the flight.")

print([(token.text, token.is_stop) for token in stop_doc])
kept_tokens = [token.text for token in stop_doc
               if not token.is_stop and not token.is_punct]
print("After filtering:", kept_tokens)


### Reflect

- What meaning is weakened or lost after filtering?
- In what kind of task might keeping stop words be important?


## 5. Stemming versus lemmatization

Both methods reduce variation, but they solve the problem differently:

| Method | How it works | Typical result |
|---|---|---|
| **Stemming** | Applies character-cutting rules without using sentence context | Fast, but may return a non-word such as `studi` |
| **Lemmatization** | Uses vocabulary and linguistic analysis | Usually returns a dictionary form such as `study` |

spaCy provides lemmatization but not stemming. The next cell uses the lightweight English Snowball stemmer for a short comparison.


In [ ]:
import snowballstemmer

stemmer = snowballstemmer.stemmer("english")
morphology_doc = nlp(
    "The researchers studied several studies and were studying better methods."
)

print(f"{'word':<14} {'stem':<14} {'lemma':<14}")
print("-" * 42)
for token in morphology_doc:
    if token.is_alpha:
        stem = stemmer.stemWord(token.text.lower())
        print(f"{token.text:<14} {stem:<14} {token.lemma_:<14}")


### Reflect

- Find a stem that is not a normal English word. How does its lemma differ?
- Which representation would be easier to show to a reader?
- Why might a fast, rough stem still be useful for a search system?


## 6. Part-of-speech tagging and ambiguity

A part-of-speech (POS) tag describes how a token functions in context. The spelling alone is not enough: the same word form can act as different parts of speech.

Predict the POS tag for `book` in each sentence before running the next cell. Then use the third sentence as a probe: does adding one more context word change the model's decision?


In [ ]:
pos_examples = [
    "I read a book.",
    "Book the flight.",
    "Please book the flight.",
]

for text in pos_examples:
    print(text)
    doc = nlp(text)
    for token in doc:
        if not token.is_punct:
            explanation = spacy.explain(token.pos_) or ""
            print(f"  {token.text:<8} {token.pos_:<6} {explanation}")
    print()


### Reflect

- Linguistically, `book` is a noun in the first required sentence and a verb in the second. Did the small model agree?
- Does adding `Please` change spaCy's tag? What does this reveal about context and model errors?
- Why would token text alone be insufficient for a reliable POS decision?


## Optional extension: dependency parsing and displaCy

Dependency parsing describes grammatical relationships between tokens, such as which noun is the subject of a verb. This is useful, but it goes beyond today's core focus on token boundaries and token attributes.

If time permits, run the next cell. The printed labels show each token's dependency role, and displaCy turns those relationships into an interactive visual inside Jupyter or Colab.


In [ ]:
from spacy import displacy

dependency_doc = nlp("Book the flight after you read the schedule.")
for token in dependency_doc:
    print(f"{token.text:<10} dependency={token.dep_:<8} head={token.head.text}")

displacy.render(
    dependency_doc,
    style="dep",
    jupyter=True,
    options={"distance": 90},
)


### Optional reflection

- Which token is the head of `flight`?
- How is the dependency label different from a POS tag?


## 7. Word-oriented tokens versus modern subword tokens

spaCy is **word-oriented**: its language rules aim to produce units that support linguistic tasks such as POS tagging and dependency parsing.

Many modern language models instead use a **subword tokenizer**. A learned vocabulary keeps frequent patterns together and divides rarer forms into smaller pieces. This limits vocabulary size and lets a model represent unfamiliar words. Subword pieces may include leading spaces and are not intended to be linguistic words.

The example below uses the `cl100k_base` byte-pair encoding available in `tiktoken`. Exact pieces depend on the tokenizer's learned vocabulary, so another model may split the same text differently.


In [ ]:
comparison_text = "Tokenization helps with antidisestablishmentarianism."

spacy_pieces = [token.text for token in nlp(comparison_text)]

encoding = tiktoken.get_encoding("cl100k_base")
subword_ids = encoding.encode(comparison_text)
subword_pieces = [encoding.decode([token_id]) for token_id in subword_ids]

print("Text:", comparison_text)
print("spaCy tokens:", spacy_pieces)
print("Subword pieces:", subword_pieces)
print("Subword token IDs:", subword_ids)


### Reflect

- Which tokenizer keeps the long word as one token, and which splits it into pieces?
- Which output is designed to support POS tags directly?
- Why should you avoid comparing token counts across models without naming the tokenizer?


## Takeaways

- Tokenization is a design decision, not simply splitting text on spaces.
- spaCy preserves some structured forms—such as abbreviations, URLs, email addresses, and decimal numbers—while separating many punctuation marks and contraction pieces.
- Lemmas and POS tags add context-sensitive linguistic information to tokens; stop-word flags are task-dependent hints.
- Stems are rule-cut forms, while lemmas aim to be meaningful base forms.
- Modern language-model tokenizers often use subword pieces optimized for a learned vocabulary rather than linguistic word boundaries.
